# __1. Euclid Flagship - Caracterização dos dados__

__Descrição:__  Caracterização dos dados da simulação Flagship 2 (FS2) (https://www.aanda.org/articles/aa/full_html/2025/05/aa50853-24/aa50853-24.html), um catálogo de galáxias desenvolvido para a missão Euclid. Trata-se de uma simulação de N-corpos executada com quatro biliões de partículas num volume cúbico de $3600 h ^{−1}$ Mpc, apresentando uma massa de partícula de aproximadamente $1,0×10^{9} M_{⊙}/h$. 

Possui um total de **4.835.535.756** galáxias até ao limite de fluxo da banda H do infravermelho próximo ($H_{E}<26,6$), sem considerar linhas de emissão ou extinção da Via Láctea. Ao aplicar-se um corte de magnitude de $H_{E≤}26$, onde a amostra é considerada completa, o catálogo retém cerca de 3,3 mil milhões de galáxias.

Geograficamente, a simulação cobre um octante do céu (aproximadamente 5.157 graus quadrados), centrada no Polo Norte Galáctico ($145^{∘}<RA<235^{∘}$ e $0^{∘}<DEC<90^{∘}$), e abrange uma vasta amplitude de redshift, de $0<z<3$. Adicionalmente, existe uma sub-região específica ($150^{∘}<RA<155^{∘}$ e $5^{∘}<DEC<10^{∘}$) que não possui cortes de magnitude ou fluxo, contribuindo com cerca de 117 milhões de objectos para o total.
 
A simulação utiliza a cosmologia de referência da missão Euclid, definida pelos seguintes parâmetros: $Ω_{m}=0.319, Ω_{b}=0.049, Ω_{Λ}+Ω_{γ}=0.681, A_{s}=2,1×10^{−9}, n_{s}=0.96$ e $h=0.67$.

Para cada uma das galáxias, são fornecidos centenas de parâmetros, incluindo:
- Informação espectroscópica e fotométrica: Fluxos em diversas bandas e distribuições de energia espectral (SEDs)
- Propriedades de lenteamento: Convergência, cisalhamento (shear) e deflexão, calculados de forma consistente com a distribuição de matéria escura
- Parâmetros de forma: Morfologia detalhada, incluindo perfis de Sérsic para componentes de bojo e disco, além de tamanhos e orientações

__Acesso aos dados:__ O acesso aos dados da simulação são feitos pelo CosmoHub (https://cosmohub.pic.es/catalogs/353). 



 # __2. Bibliotecas__

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sn
import cartopy.crs as ccrs

# __3. Seleção dos dados__

Devido à alta volumetria, o CosmoHub faz uma seleção aleatória defrações dos dados, através da divisão em buckets. Foi aplicado um fator de redução de **1:256**, obtendo uma amostra de aproximadamente **18,9 milhões de objetos**, correspondente a **0,39%** do conjunto de dados original.

Desse conjunto, foram selecionadas aleratoriamente **100000 galáxias** para a caracterização representativa do conjunto total.

Descrição das colunas selecionadas:
- __Identificação e posição__
    - __galaxy_id__: Identificador único atribuído a cada uma das galáxias no catálogo
    - __kind__: Indica o tipo de galáxia, diferenciando se ela é uma central (posicionada no centro do halo) ou uma satélite (distribuída conforme um perfil NFW triaxial)
    - __ra_gal / dec_gal__: Coordenadas de Ascensão Reta e Declinação, que definem a posição angular da galáxia no céu (um octante na FS2)
    - __true_redshift_gal__: Redshift cosmológico real da galáxia, derivado da sua distância comóvel na simulação antes de considerar efeitos de velocidade peculiar
- __Fotometria__
    - __Bandas Euclid__: vis (visível) e y, j, h (infravermelho próximo)
    - __Bandas LSST__: u, g, r, i, z, y
    - __Tipos de fluxo__:
        - __abs__: fluxos no referencial da galáxia a uma distância de 10 pc
        - __el_model3__: Indica que os fluxos incluem a contribuição de linhas de emissão (el), calibradas de acordo com o Modelo 3 de Pozzetti et al. (2016)
        - __ext_odonnell_ext__: Indica que o valor foi corrigido pela extinção por poeira, utilizando a lei de O'Donnell (1994)
        - __error__: Erro fotométrico estimado para a observação
        - __error_realization__: Uma realização aleatória do fluxo/magnitude baseada no erro fotométrico, simulando uma observação realista
- __Morfologia__
    -  __dominant_shape__: Define o modelo da galáxia: 0 para apenas bojo (Sérsic puro) e 1 para bojo + disco
    -  __bulge_fraction__: Razão entre o fluxo do bojo e o fluxo total (B/T)
    -  __inclination_angle__: Ângulo de inclinação da galáxia, variando de 0° (face-on) a 90° (edge-on)
    -  __median_major_axis__: Comprimento de escala mediano do semieixo maior, usado como base para as dimensões da galáxia
    -  __scale_length__: Valor de referência para os comprimentos de escala do disco e do bojo
    -  __eps1_gal / eps2_gal__: Componentes da elipsicidade intrínseca da galáxia em 2D
    -  __Disco__
        -  __disk_r50__: Raio de meia-luz do componente de disco
        -  __disk_scalelength__: Comprimento de escala (scale-length) do perfil exponencial do disco
        -  __disk_nsersic__: Índice de Sérsic do disco (fixado em 1 para perfis exponenciais)
        -  __disk_angle__: Posição do eixo de rotação do disco
        -  __disk_ellipticity / disk_axis_ratio__: Elipsicidade e razão de eixos (b/a) projetada do disco
    -  __Bojo__
        -  __bulge_r50__: Raio de meia-luz do componente de bojo
        -  __bulge_nsersic__: Índice de Sérsic do bojo, representando a concentração de luz (calibrado entre 0.5 e 6.0)
        -  __bulge_ellipticity / bulge_axis_ratio__: Elipsicidade e razão de eixos projetada do bojo
    

 


In [ ]:
data_set = pd.read_parquet('../Dados/euclid_flagship_cut.parquet')

data_set.columns

# __4. Caracterização Geral__

## 4.1 Área do céu

In [ ]:
ra = data_set["ra_gal"].to_numpy()
dec = data_set["dec_gal"].to_numpy()


ra = 180 - ra
ra = (ra + 180) % 360 - 180

bins_ra = np.linspace(-180, 180, 361)
bins_dec = np.linspace(-90, 90, 181)

H, xedges, yedges = np.histogram2d(ra,dec,bins=(bins_ra, bins_dec))
H = H.astype(float)
H[H == 0] = np.nan


fig = plt.figure(figsize=(14, 7))
ax = plt.axes(projection=ccrs.Mollweide())


mesh = ax.pcolormesh(xedges,yedges,H.T,transform=ccrs.PlateCarree(),cmap="viridis")


gl = ax.gridlines(crs=ccrs.PlateCarree(),draw_labels=False,linewidth=0.7,linestyle="--",color="gray",alpha=0.6)
gl.xlocator = mticker.FixedLocator([-150,-120,-90,-60,-30,0,30,60,90,120,150])
gl.ylocator = mticker.FixedLocator([-75,-60,-45,-30,-15,0,15,30,45,60,75])


for x in [-150,-120,-90,-60,-30,0,30,60,90,120,150]:
    ra_label = (180 - x) % 360
    ax.text(x,-2,f"{int(ra_label)}°",transform=ccrs.PlateCarree(),ha="center",va="top",fontsize=10)
    
for y in [-75,-60,-45,-30,-15,0,15,30,45,60,75]:
    ax.text(-178,y,f"{y}°",transform=ccrs.PlateCarree(),ha="right",va="center",fontsize=10)

cbar = plt.colorbar(mesh,ax=ax,shrink=0.8,pad=0.05)
cbar.set_label("Número de objetos")

plt.title("Distribuição Espacial\n(Projeção de Mollweide)",fontsize=16)

plt.savefig('../Figuras/area_ceu.pdf', format='pdf',bbox_inches='tight',dpi=300)
plt.show()

## 4.2 Distribuição de redshift

In [ ]:
def z_hist(catalog, z_col, sigma=0, pop='', save=False):

    z = catalog[z_col].dropna()

    fig, ax = plt.subplots(figsize=(8, 5))

    ax.hist(z,bins=100,color='cornflowerblue',edgecolor='black',linewidth=0.4,alpha=0.8)

    ax.set_xlabel(r'Redshift ($z$)', fontsize=15)
    ax.set_ylabel('Número de galáxias', fontsize=15)
    ax.set_title('Distribuição de redshifts', fontsize=17)

    ax.grid(linestyle='--',linewidth=0.6,alpha=0.35)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(axis='both', labelsize=13)

    text = (f'N = {len(z):,}\n'f'⟨z⟩ = {z.mean():.3f}\n'f'σ = {z.std():.3f}')

    ax.text(0.98,0.97,text,transform=ax.transAxes,ha='right',va='top',fontsize=11,bbox=dict(facecolor='white',edgecolor='gray',alpha=0.85,boxstyle='round'))

    plt.tight_layout()

 
    plt.savefig(f'../Figuras/z_hist{pop}.pdf',dpi=300,bbox_inches='tight')

    plt.show()

In [ ]:
z_hist(data_set, 'true_redshift_gal')

## 4.3 Fotometria

## Bandas fotométricas

### Bandas Euclid
(<https://www.euclid-ec.org/science/overview/>)

O satélite **Euclid** realiza observações fotométricas em quatro bandas, utilizando dois instrumentos:

- **VIS**, responsável pela banda no visível
    - $I_{E}: 530{-}920\,\mathrm{nm}$

- **NISP**, responsável pelas três bandas no infravermelho próximo
    - $Y_{E}: 949.6{-}1212.3\,\mathrm{nm}$
    - $J_{E}: 1167.6{-}1567.0\,\mathrm{nm}$
    - $H_{E}: 1521.5{-}2021.4\,\mathrm{nm}$

As magnitudes-limite do **Euclid Wide Survey** (sistema AB, S/N = 5 para fontes pontuais) são:

- $I_{E}=26.2$
- $Y_{E}=24.3$
- $J_{E}=24.5$
- $H_{E}=24.4$

---

### Bandas LSST
(<https://www.lsst.org/sites/default/files/docs/sciencebook/SB_2.pdf>, p. 34)

O sistema fotométrico do **LSST** é composto por seis filtros ópticos ($u$, $g$, $r$, $i$, $z$ e $y$), cobrindo desde o ultravioleta próximo até o infravermelho próximo.

Os intervalos aproximados de transmissão são:

- $u: 320{-}400\,\mathrm{nm}$
- $g: 400{-}552\,\mathrm{nm}$
- $r: 552{-}691\,\mathrm{nm}$
- $i: 691{-}818\,\mathrm{nm}$
- $z: 818{-}922\,\mathrm{nm}$
- $y: 950{-}1080\,\mathrm{nm}$

---

## Fluxos e erros fotométricos

No catálogo **Euclid Flagship**, os fluxos fotométricos são fornecidos em unidades de fluxo espectral,

$$
\mathrm{erg\,cm^{-2}\,s^{-1}\,Hz^{-1}}.
$$

O catálogo disponibiliza, para cada banda do **Euclid** e do **LSST**:

- o fluxo modelado (intrínseco);
- a incerteza fotométrica ($1\sigma$);
- uma realização observacional do fluxo, obtida após a aplicação do modelo de ruído.

De acordo com a documentação do catálogo, o ruído fotométrico é aplicado **apenas às bandas observacionais do Euclid e do LSST**, sendo calibrado para reproduzir a profundidade esperada do **Euclid Wide Survey (DR3)**.

As profundidades adotadas na simulação correspondem a limites de **10σ** (fonte pontual em abertura de $2''$):

| Banda | Magnitude-limite (10σ) |
|:------:|:----------------------:|
| $u$ | 24.4 |
| $g$ | 25.6 |
| $r$ | 25.7 |
| $i$ | 25.0 |
| $z$ | 24.3 |
| $y$ | 24.3 |
| $I_E$ | 25.0 |
| $Y_E$ | 23.5 |
| $J_E$ | 23.5 |
| $H_E$ | 23.5 |

Assim, o fluxo observado pode ser representado por

$$
F_{\rm obs}=F_{\rm modelo}+F_{\rm erro},
$$

onde $F_{\rm erro}$ representa uma realização do ruído fotométrico compatível com a profundidade prevista para cada banda.

As magnitudes no sistema AB são obtidas a partir do fluxo observado segundo

$$
m_{\rm AB}=-2.5\log_{10}(F_\nu)-48.6,
$$

onde $F_\nu$ é o fluxo espectral observado na respectiva banda fotométrica.

### 4.3.1 Gerando os fluxos observados


In [ ]:
bands_lsst = ['u', 'g', 'r', 'i', 'z', 'y']
bands_euclid = ['nisp_j','nisp_y', 'nisp_h','vis']

In [ ]:
for j in bands_euclid:
    data_set[f'flux_observed_euclid_{j}'] = (data_set[f'euclid_{j}_el_model3_ext_odonnell_ext']
        + data_set[f'euclid_{j}_el_model3_ext_odonnell_ext_error_realization'])
    
    data_set[f'fluxerr_observed_euclid_{j}'] = (data_set[f'euclid_{j}_el_model3_ext_odonnell_ext_error'])

for j in bands_lsst:
    data_set[f'flux_observed_lsst_{j}'] = (data_set[f'lsst_{j}_el_model3_ext_odonnell_ext']
        + data_set[f'lsst_{j}_el_model3_ext_odonnell_ext_error_realization'])
    
    data_set[f'fluxerr_observed_lsst_{j}'] = (data_set[f'lsst_{j}_el_model3_ext_odonnell_ext_error'])

In [ ]:
data_set.columns

### 4.3.2 Conversão de fluxos em magnitudes

In [ ]:
for j in bands_euclid:
    data_set[f'mag_euclid_{j}'] = (-2.5 * np.log10(data_set[f'flux_observed_euclid_{j}']) - 48.6)

for j in bands_lsst:
    data_set[f'mag_lsst_{j}'] = (-2.5 * np.log10(data_set[f'flux_observed_lsst_{j}']) - 48.6)

### 4.3.3 Erros das magnitudes

Propagando o erro da magnitude, temos: $\sigma_{mag} = 1.085736 \frac{\sigma_{flux}}{flux}$

In [ ]:
for j in bands_euclid:
    data_set[f'magerr_euclid_{j}'] = (1.085736205 * data_set[f'fluxerr_observed_euclid_{j}'] /data_set[f'flux_observed_euclid_{j}'])

for j in bands_lsst:
    data_set[f'magerr_lsst_{j}'] = (1.085736205 * data_set[f'fluxerr_observed_lsst_{j}'] /data_set[f'flux_observed_lsst_{j}'])

In [ ]:
data_set.columns

### 4.3.4 Distribuição de magnitudes Euclid

In [ ]:
def magnitude_distributions(catalog, bands, survey='', save=False):

    fig, ax = plt.subplots(figsize=(8, 5))

    colors = plt.cm.tab10.colors

    for i, band in enumerate(bands):

        ax.hist(catalog[f'mag_{survey}_{band}'],bins=50,histtype='step',linewidth=2,alpha=0.8,color=colors[i % len(colors)],label=band)

    ax.set_yscale('log')
    ax.set_xlabel('Magnitude', fontsize=14)
    ax.set_ylabel('Número de galáxias', fontsize=14)
    ax.set_title(f'Distribuição de magnitudes - {survey}', fontsize=16)
    ax.tick_params(axis='both', labelsize=12)
    ax.grid(linestyle='--',alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.legend(fontsize=10,ncol=2,loc='upper right')

    plt.tight_layout()

  
    plt.savefig(f'../Figuras/magnitude_distributions_{survey}.pdf',dpi=300,bbox_inches='tight')

    plt.show()

In [ ]:
def mag_err(catalog, survey):
    plt.figure(figsize=(25, 25))
    bands_lsst = ['u','g', 'r', 'i', 'z','y']
    bands_euclid = ['nisp_j', 'nisp_y', 'nisp_h', 'vis']
  
    if survey == 'lsst':
        bands = bands_lsst
    elif survey == 'euclid':
        bands = bands_euclid
    else:
        raise ValueError(f"Survey desconhecido: {survey}")
 
    for i, band in enumerate(bands, 1):
        plt.subplot(2, 2, i)
        query = f'magerr_{survey}_{band} < 2.1'
        data = catalog.query(query)
        mag = np.array(data[f'mag_{survey}_{band}'])
        err = np.array(data[f'magerr_{survey}_{band}'])
    
        hb = plt.hexbin(mag, err, gridsize=80, cmap='GnBu', bins='log', mincnt=1)
    
        plt.xlabel(f"mag {band}", fontsize=25)
        if i == 1:
            plt.ylabel("error", fontsize=25)
        plt.xlim(14,30)
        plt.ylim(0,2.1)
        plt.grid(True, linestyle='--', alpha=0.6)
        cbar = plt.colorbar(hb, label='log(N)')
        cbar.set_label('log(N)', fontsize=20)
        cbar.ax.tick_params(labelsize=20)
        plt.tick_params(axis='both', labelsize=20)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig('../Figuras/mag_err.pdf', format='pdf', bbox_inches='tight', dpi=300)
    plt.show()

In [ ]:
def mag_vs_err_all_bands(catalog, survey):
    bands_lsst = ['u', 'g', 'r', 'i', 'z', 'y']
    bands_euclid = ['nisp_j', 'nisp_y', 'nisp_h', 'vis']
    
    colors_lsst = ['#9467bd', '#1f77b4', '#2ca02c', '#ff7f0e', '#d62728', '#8c564b']
    colors_euclid = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
    
    if survey == 'lsst':
        bands = bands_lsst
        colors = colors_lsst
    elif survey == 'euclid':
        bands = bands_euclid
        colors = colors_euclid
    else:
        raise ValueError(f"Survey desconhecido: {survey}")
    
    plt.figure(figsize=(10, 7), dpi=150)
    
    for band, color in zip(bands, colors):
        query = f'magerr_{survey}_{band} < 2.1'
        data = catalog.query(query)
        
        mag = np.array(data[f'mag_{survey}_{band}'])
        err = np.array(data[f'magerr_{survey}_{band}'])
        
        plt.scatter(mag, err, alpha=0.1, s=3, color=color, label=f'Banda {band}')

    plt.xlabel("Magnitude", fontsize=16)
    plt.ylabel("Erro de Magnitude", fontsize=16)
    plt.title(f"Magnitude vs. Erro de Magnitude ({survey.upper()})", fontsize=18, pad=12)
    plt.xlim(14, 30)
    plt.ylim(0, 2.1)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tick_params(axis='both', labelsize=14)
    
    leg = plt.legend(loc='upper left', fontsize=12, frameon=True)
    for lh in leg.legend_handles:
        lh.set_alpha(1.0)
        
    plt.tight_layout()
    plt.savefig(f'../Figuras/mag_vs_err_{survey}.pdf', format='pdf', bbox_inches='tight', dpi=300)
    plt.show()

In [ ]:
magnitude_distributions(data_set, bands_euclid, survey="euclid", save=True)

In [ ]:
mag_err(data_set, 'euclid')

In [ ]:
mag_vs_err_all_bands(data_set, 'euclid')

### 4.3.5 Distribuição de magnitudes LSST

In [ ]:
magnitude_distributions(data_set, bands_lsst, survey="lsst", save=True)

In [ ]:
mag_err(data_set, 'lsst')

In [ ]:
mag_vs_err_all_bands(data_set, 'lsst')

### 4.3.6 Comparação banda VIS com bandas LSST

In [ ]:
def magnitude_distribution_vis_lsst(catalog, save=False):

    bands = ['u', 'g', 'r', 'i', 'z', 'y']
    colors = {'u': 'purple','g': 'blue','r': 'green','i': 'orange','z': 'red','y': 'brown'}

    fig, axes = plt.subplots(2, 3,figsize=(14, 8),sharex=True,sharey=True)
    axes = axes.ravel()

    for ax, band in zip(axes, bands):
        ax.hist(catalog['mag_euclid_vis'],bins=50,histtype='stepfilled',linewidth=2.5,color='lightblue',label='Euclid VIS')

        ax.hist(catalog[f'mag_lsst_{band}'],bins=50,histtype='step',linewidth=2,color=colors[band],label=f'LSST {band}')

        ax.set_title(f'VIS × {band}', fontsize=14)
        ax.set_yscale('log')
        ax.grid(alpha=0.3, linestyle='--')
        ax.legend(fontsize=10)

    fig.supxlabel('Magnitude', fontsize=15)
    fig.supylabel('Número de galáxias', fontsize=15)
    fig.suptitle('Comparação entre a banda VIS Euclid e as bandas LSST',fontsize=18,y=1.02)
    plt.tight_layout()

    if save:
        plt.savefig('../Figuras/magnitude_distributions_VIS_LSST.pdf',dpi=300,bbox_inches='tight')

    plt.show()

In [ ]:
magnitude_distribution_vis_lsst(data_set, save=True)

### 4.3.7 Diagramas cor-magnitude (absoluta)

O catálogo fornece os fluxos absolutos, de modo que as magnitudes absolutas são obtidas por:

$$
M_{\mathrm{AB}} = -2.5 \log_{10}(F_{\mathrm{abs}}) - 48.6 + 5 \log_{10}(h),
$$

onde $F_{\mathrm{abs}}$ é o fluxo absoluto fornecido pelo catálogo e $h = 0.67$ é o parâmetro adimensional de Hubble adotado na simulação.

#### 4.3.7.1. Convsersão de fluxos para magnitudes

In [ ]:
h = 0.67

for j in bands_euclid:
    data_set[f'mag_abs_euclid_{j}'] = (-2.5 * np.log10(data_set[f"euclid_{j}_abs"])- 48.6 + 5 * np.log10(h))

for j in bands_lsst:
    data_set[f'mag_abs_lsst_{j}'] = (-2.5 * np.log10(data_set[f"lsst_{j}_abs"])- 48.6 + 5 * np.log10(h))

In [ ]:
data_set.columns

#### 4.3.7.2 Magnitude absoluta: banda r LSST

In [ ]:
def color_magnitude_diagram(catalog,mag_band,mag_survey,color_band1,color_band2,color_survey,figsize=(8,7)):

    mag_col = f"mag_abs_{mag_survey}_{mag_band}"
    col1 = f"mag_abs_{color_survey}_{color_band1}"
    col2 = f"mag_abs_{color_survey}_{color_band2}"

    M = catalog[mag_col].values
    M1 = catalog[col1].values
    M2 = catalog[col2].values
    z = catalog["true_redshift_gal"].values

    color = M1 - M2

    fig, ax = plt.subplots(figsize=figsize,dpi=120)

    scatter = ax.scatter(M,color,c=z,cmap="jet",s=2,alpha=0.8,rasterized=True)

    ax.set_xlabel(rf"$M_{{{mag_band}}}^{{{mag_survey.upper()}}}$",fontsize=15)
    ax.set_ylabel(rf"${color_band1}-{color_band2}$ ({color_survey.upper()})",fontsize=15)
    ax.set_title(f"{color_band1}-{color_band2} vs M{mag_band}",fontsize=15)
    ax.grid(alpha=0.2)

    cbar = plt.colorbar(scatter,ax=ax)
    cbar.set_label("Redshift",fontsize=13)

    plt.tight_layout()
    plt.savefig(f'../Figuras/cor_magnitude_{color_survey}_{color_band1}-{color_band2}_M{mag_band}_{mag_survey}.pdf',dpi=300,bbox_inches='tight')
    plt.show()

In [ ]:
color_magnitude_diagram(data_set,'r', 'lsst', 'g', 'r', 'lsst')

#### 4.3.7.3 Magnitude absoluta: banda i LSST

In [ ]:
color_magnitude_diagram(data_set,'i', 'lsst', 'r', 'i', 'lsst')

#### 4.3.7.4 Magnitude absoluta: banda vis Euclid

In [ ]:
color_magnitude_diagram(data_set,'vis', 'euclid', 'vis', 'nisp_h', 'euclid')

### 4.3.8 Diagramas cor-magnitude (aparente)

In [ ]:
def color_magnitude_apparent(catalog,mag_band,color_band1,color_band2,mag_survey="lsst",color_survey="lsst",sample=200000,figsize=(8,7)):

    mag_col = f"mag_{mag_survey}_{mag_band}"

    color_col1 = f"mag_{color_survey}_{color_band1}"
    color_col2 = f"mag_{color_survey}_{color_band2}"

    M = catalog[mag_col].values
    color = catalog[color_col1].values - catalog[color_col2].values
    z = catalog["true_redshift_gal"].values

    mask = np.isfinite(M) & np.isfinite(color) & np.isfinite(z)

    M = M[mask]
    C = color[mask]
    z = z[mask]

    if sample is not None and len(M) > sample:
        idx = np.random.choice(len(M),sample,replace=False)

        M = M[idx]
        C = C[idx]
        z = z[idx]

    fig, ax = plt.subplots(figsize=figsize,dpi=120)

    scatter = ax.scatter(C,M,c=z,cmap="jet",s=2,alpha=0.8,rasterized=True)

    ax.set_ylabel(rf"$m_{{{mag_band}}}$",fontsize=15)
    ax.set_xlabel(rf"${color_band1}-{color_band2}$",fontsize=15)

    ax.set_title(f"{color_band1}-{color_band2} vs $m_{{{mag_band}}}$",fontsize=15)

    ax.grid(alpha=0.2)

    cbar = plt.colorbar(scatter,ax=ax)
    cbar.set_label("Redshift",fontsize=13)

    plt.tight_layout()

    plt.savefig(f"../Figuras/CMD_apparent_{mag_survey}_{mag_band}_{color_survey}_{color_band1}-{color_band2}.pdf",dpi=300,bbox_inches="tight")

    plt.show()

#### 4.3.8.1 Diagrama mag r LSST

In [ ]:
color_magnitude_apparent(data_set, 'r','g', 'r', 'lsst','lsst')

#### 4.3.8.2 Diagrama banda i LSST

In [ ]:
color_magnitude_apparent(data_set, 'i','r', 'i', 'lsst','lsst')

#### 4.3.8.3 Diagrama banda vis Euclid

In [ ]:
color_magnitude_apparent(data_set, 'vis','vis', 'nisp_h', 'euclid','euclid')

### 4.3.9 Diagrama cor-cor com z

In [ ]:
def color_color_redshift(catalog, survey, sigma=0, pop='', save=0):

    if survey.lower() == 'lsst':
        bands = bands_lsst
        nrows, ncols = 2, 2  # 4 diagramas

    elif survey.lower() == 'euclid':
        bands = bands_euclid
        nrows, ncols = 2, 2  # 1 diagrama

    else:
        raise ValueError("survey deve ser 'lsst' ou 'euclid'")

    plt.figure(figsize=(12, 12))

    i = 1

    for index in range(len(bands) - 2):

        plt.subplot(nrows, ncols, i)
        i += 1

        color = catalog[f'mag_{survey}_{bands[index+1]}']
        next_color = catalog[f'mag_{survey}_{bands[index+2]}']
        past_color = catalog[f'mag_{survey}_{bands[index]}']

        hb = plt.hexbin(
            color - next_color,
            past_color - color,
            C=catalog['true_redshift_gal'],
            reduce_C_function=np.median,
            mincnt=5,
            cmap='jet',
            gridsize=150
        )

        plt.xlabel(
            f'{bands[index+1]} - {bands[index+2]}',
            fontsize=16
        )

        plt.ylabel(
            f'{bands[index]} - {bands[index+1]}',
            fontsize=16
        )

        #plt.xlim(-2, 2)
        #plt.ylim(-2, 2)

        cbar = plt.colorbar(hb)
        cbar.set_label('Median Redshift', fontsize=14)
        cbar.ax.tick_params(labelsize=12)

        plt.tick_params(axis='both', labelsize=12)

    plt.tight_layout()
    plt.savefig(f'../Figuras/color_color_redshift_{survey}.pdf',bbox_inches='tight',dpi=300)
    plt.show()

#### 4.3.9.1. LSST bands

In [ ]:
color_color_redshift(data_set, 'lsst')

#### 4.3.9.1. Euclid bands

In [ ]:
color_color_redshift(data_set, 'euclid')

### 4.3.10 Redshift x cor

In [ ]:
def redshift_vs_color(catalog, survey):
    bands_lsst = ['u', 'g', 'r', 'i', 'z', 'y']
    bands_euclid = ['vis', 'nisp_y', 'nisp_j', 'nisp_h']
    
    if survey == 'lsst':
        bands = bands_lsst
        nrows, ncols = 2, 3
        figsize = (22, 12)
    elif survey == 'euclid':
        bands = bands_euclid
        nrows, ncols = 1, 3
        figsize = (22, 5.5)
    else:
        raise ValueError(f"Survey desconhecido: {survey}")
    
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes = np.array(axes).flatten()
    
    color_pairs = [(bands[i], bands[i+1]) for i in range(len(bands)-1)]
    
    for i, (b1, b2) in enumerate(color_pairs):
        ax = axes[i]
  
        query = f'magerr_{survey}_{b1} < 2.1 and magerr_{survey}_{b2} < 2.1'
        data = catalog.query(query)
        
        z = np.array(data['true_redshift_gal'])
        color = np.array(data[f'mag_{survey}_{b1}']) - np.array(data[f'mag_{survey}_{b2}'])
        
        hb = ax.hexbin(z, color, gridsize=80, cmap='GnBu', bins='log', mincnt=1)
        
        ax.set_xlabel("redshift", fontsize=20)
        ax.set_ylabel(f"{b1} - {b2}", fontsize=20)
        ax.set_xlim(0, 3.0)
        ax.set_ylim(-1.5, 3.5)
        ax.grid(True, linestyle='--', alpha=0.6)
        ax.tick_params(axis='both', labelsize=16)
        
        cbar = fig.colorbar(hb, ax=ax)
        cbar.set_label('log(N)', fontsize=16)
        cbar.ax.tick_params(labelsize=14)
        
    for j in range(len(color_pairs), len(axes)):
        fig.delaxes(axes[j])
        
    plt.tight_layout()
    plt.savefig(f'../Figuras/redshift_vs_color_{survey}.pdf', format='pdf', bbox_inches='tight', dpi=300)
    plt.show()

In [ ]:
redshift_vs_color(data_set, 'euclid')

In [ ]:
redshift_vs_color(data_set, 'lsst')

### 4.3.11 Redshift x magnitude

In [ ]:
def redshift_vs_mag(catalog, survey):
    bands_lsst = ['u', 'g', 'r', 'i', 'z', 'y']
    bands_euclid = ['vis', 'nisp_y', 'nisp_j', 'nisp_h']
    
    if survey == 'lsst':
        bands = bands_lsst
        nrows, ncols = 2, 3
        figsize = (22, 12)  
    elif survey == 'euclid':
        bands = bands_euclid
        nrows, ncols = 1, 4  
        figsize = (22, 5.5) 
    else:
        raise ValueError(f"Survey desconhecido: {survey}")
    
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes = np.array(axes).flatten()
    
    for i, band in enumerate(bands):
        ax = axes[i]
        
        query = f'magerr_{survey}_{band} < 2.1'
        data = catalog.query(query)
        
        z = np.array(data['true_redshift_gal']) 
        mag = np.array(data[f'mag_{survey}_{band}'])
        
        hb = ax.hexbin(z, mag, gridsize=80, cmap='GnBu', bins='log', mincnt=1)
        
        ax.set_xlabel("redshift", fontsize=18)
        ax.set_ylabel(f"mag {band}", fontsize=18)
        ax.set_xlim(0, 3.0)
        ax.set_ylim(14, 30)
        ax.grid(True, linestyle='--', alpha=0.6)
        ax.tick_params(axis='both', labelsize=14)
        
        cbar = fig.colorbar(hb, ax=ax, pad=0.02)
        cbar.set_label('log(N)', fontsize=14)
        cbar.ax.tick_params(labelsize=12)

    for j in range(len(bands), len(axes)):
        fig.delaxes(axes[j])
        
    plt.tight_layout()
    plt.savefig(f'../Figuras/redshift_vs_mag_{survey}.pdf', format='pdf', bbox_inches='tight', dpi=300)
    plt.show()

In [ ]:
redshift_vs_mag(data_set, 'euclid')

In [ ]:
redshift_vs_mag(data_set, 'lsst')

## 4.4 Morforlogia

### 4.4.1 Figura 20: fração de galáxias modeladas com 1 componente

In [ ]:
from scipy.signal import savgol_filter
def bulge_fraction_vs_i(catalog,mag_col="mag_lsst_i",shape_col="dominant_shape",bins=30):

    mag = catalog[mag_col].values
    shape = catalog[shape_col].values

    mask = np.isfinite(mag) & np.isfinite(shape)

    mag = mag[mask]
    shape = shape[mask]

    bins_mag = np.linspace(19,26,bins+1)

    frac = []
    mag_center = []

    for i in range(len(bins_mag)-1):

        idx = (mag >= bins_mag[i]) & (mag < bins_mag[i+1])

        total = np.sum(idx)

        if total > 0:
            bulge = np.sum(shape[idx] == 0)

            frac.append(bulge/total)
            mag_center.append((bins_mag[i]+bins_mag[i+1])/2)

    mag_center = np.array(mag_center)
    frac = np.array(frac)

    frac_smooth = savgol_filter(frac,7,2)


    fig, ax = plt.subplots(figsize=(7,5),dpi=120)

    ax.plot(mag_center,frac_smooth,linewidth=2,label="Flagship", color='red')
    ax.set_xlabel(r"$i$-band magnitude",fontsize=14)
    ax.set_ylabel("Fraction",fontsize=14)

    ax.set_xlim(19,26)
    ax.set_ylim(0,0.45)

    ax.grid(alpha=0.3)

    ax.legend()

    plt.tight_layout()

    plt.savefig("../Figuras/bulge_fraction_i_band.pdf",dpi=300,bbox_inches="tight")

    plt.show()

In [ ]:
bulge_fraction_vs_i(data_set)

### 4.4.2 Figura 22: Distribuição da fração do bojo

In [ ]:
catalog = data_set

mask = catalog['mag_euclid_vis'] < 24.5
bt = catalog[mask]['bulge_fraction']

bt = bt[np.isfinite(bt)]
bt = bt[(bt >= 0) & (bt <= 1)]

bins = np.linspace(0, 1, 20)
counts, edges = np.histogram(bt, bins=bins)

fraction = counts / counts.sum()
centers = 0.5 * (edges[:-1] + edges[1:])

def smooth(y, window=3):
    kernel = np.ones(window) / window
    return np.convolve(y, kernel, mode='same')

smooth_fraction = smooth(fraction, window=1)

plt.figure(figsize=(7,5))
plt.plot(centers, smooth_fraction, color='red') 

plt.xlabel('Bulge fraction', fontsize=14)
plt.ylabel('Fraction', fontsize=14)
plt.xlim(0,1)
#plt.ylim(0, 0.3)
plt.grid(alpha=0.3)

plt.savefig("../Figuras/bulge_fraction_distribuiion.pdf",dpi=300,bbox_inches="tight")
plt.show()

### 4.4.3 Figuras 23 e 24: Distribuição do raio de meia-luz para um e dois componentes

In [ ]:
def flagship_fig23_24(catalog,mag_col="mag_lsst_i",mag_limit=24.5,shape_col="dominant_shape",disk_r50_col="disk_r50",scale_col="scale_length",bins=50):

    fig, ax = plt.subplots(2, 1,figsize=(7, 8),dpi=120,sharex=True)

    two = catalog[(catalog[shape_col] == 1) & (catalog[mag_col] < mag_limit)]
    r_disk = two[disk_r50_col].values

    mask = np.isfinite(r_disk) & (r_disk > 0)
    x = np.log10(r_disk[mask])

    log_r_sersic_two = (-1.81+ 2.6 / (1 + np.exp(-2.15*(x + 0.3)))+ 0.7 / (1 + np.exp(-15.0*(x - 0.1))))

    ax[0].hist(log_r_sersic_two,bins=bins,density=True,histtype="step",linewidth=2,color="red",label="Flagship")
    ax[0].set_ylabel("relative abundance")
    ax[0].text(0.95,0.85,"two-component",transform=ax[0].transAxes,ha="right")
    ax[0].legend()
    ax[0].grid(alpha=0.3)

    one = catalog[(catalog[shape_col] == 0) &(catalog[mag_col] < mag_limit)]
    r_one = one[scale_col].values
    mask = np.isfinite(r_one) & (r_one > 0)
    log_r_sersic_one = np.log10(r_one[mask])


    ax[1].hist(log_r_sersic_one,bins=bins,density=True,histtype="step",linewidth=2,color="red",label="Flagship")
    ax[1].set_ylabel("relative abundance")
    ax[1].text(0.95,0.85,"one-component",transform=ax[1].transAxes,ha="right")
    ax[1].legend()
    ax[1].grid(alpha=0.3)
    ax[1].set_xlabel(r"$\log_{10}(R_{50}^{S\acute{e}rsic}/arcsec)$")

    plt.tight_layout()
    plt.show()


    print("two-component median:",np.median(log_r_sersic_two))
    print("one-component median:",np.median(log_r_sersic_one))

In [ ]:
flagship_fig23_24(data_set)

### 4.4.4 Figura 25: n sérsic

In [ ]:
def sersic_index_distribution(catalog,nsersic_col="bulge_nsersic",shape_col="dominant_shape",mag_col="mag_lsst_i",mag_limit=24.5,bins=50):

    fig,ax = plt.subplots(2,1,figsize=(7,8),dpi=120,sharex=True)

    samples = [(1,"two-component"),(0,"one-component")]

    for a,(shape,label) in zip(ax,samples):

        data = catalog[(catalog[shape_col]==shape) & (catalog[mag_col]<mag_limit)]
        n = data[nsersic_col].values
        n = n[np.isfinite(n) & (n>0)]
        a.hist(n,bins=bins,density=True,histtype="step",linewidth=2,label="Flagship", color='red')
        a.set_ylabel("relative abundance",fontsize=12)
        a.text(0.95,0.85,label,transform=a.transAxes,ha="right",fontsize=12)
        a.grid(alpha=0.3)
        a.legend()

    ax[-1].set_xlabel(r"$n_{\rm Sérsic}$",fontsize=14)
    ax[-1].set_xlim(0,8)
    
    plt.tight_layout()
    plt.savefig("../Figuras/sersic_index_distribution.pdf",dpi=300,bbox_inches="tight")
    plt.show()

In [ ]:
sersic_index_distribution(data_set)

### 4.4.5 Figura 26: ângulo de inclinação

In [ ]:
def inclination_distribution(catalog,angle_col="inclination_angle",shape_col="dominant_shape",mag_col="mag_lsst_i",mag_limit=24.5,bins=50):

    data = catalog[(catalog[shape_col]==1) & (catalog[mag_col]<mag_limit)]

    theta = data[angle_col].values
    theta = theta[np.isfinite(theta)]

    fig,ax = plt.subplots(figsize=(7,5),dpi=120)

    ax.hist(theta,bins=bins,density=True,histtype="step",linewidth=2,label="Flagship",color="red")
    ax.set_xlabel(r"$\theta_{\rm incl}$ (deg)",fontsize=14)
    ax.set_ylabel("relative abundance",fontsize=14)
    ax.set_title("Two-component galaxies, i < 24.5")
    ax.grid(alpha=0.3)
    ax.legend()

    plt.tight_layout()
    plt.savefig("../Figuras/inclination_distribution.pdf",dpi=300,bbox_inches="tight")
    plt.show()

In [ ]:
inclination_distribution(data_set)

### 4.4.6 Figura 27: Distribuição dos valores de elipticidade

In [ ]:
def ellipticity_distribution(catalog,disk_col="disk_ellipticity",bulge_col="bulge_ellipticity",shape_col="dominant_shape",mag_col="mag_lsst_i",mag_limit=24.5,bins=50):

    fig,ax = plt.subplots(3,1,figsize=(7,10),dpi=120,sharex=True)

    samples = [("disk - two-component",1,disk_col),("Sérsic bulge - two-component",1,bulge_col),("Sérsic - one-component",0,bulge_col)]

    for a,(label,shape,col) in zip(ax,samples):

        data = catalog[(catalog[shape_col]==shape) & (catalog[mag_col]<mag_limit)]

        e = data[col].values
        e = e[np.isfinite(e) & (e>=0)]
        
        a.hist(e,bins=bins,density=True,histtype="step",linewidth=2,color="red",label="Flagship")
        a.set_ylabel("relative abundance",fontsize=12)
        a.text(0.95,0.85,label,transform=a.transAxes,ha="right",fontsize=12)
        a.grid(alpha=0.3)
        a.legend()


    ax[-1].set_xlabel(r"$\epsilon$",fontsize=14)
    ax[-1].set_xlim(0,1)

    plt.tight_layout()
    plt.savefig("../Figuras/ellipticity_distribution.pdf",dpi=300,bbox_inches="tight")
    plt.show()

In [ ]:
ellipticity_distribution(data_set)

In [ ]:
data_set["sn_lsst_i"] = 1.0857 / data_set["magerr_lsst_i"]
data_set_cut = data_set[(data_set["mag_lsst_i"] < 24.5) & (data_set["sn_lsst_i"] > 5)].copy()

In [ ]:
color_magnitude_diagram(data_set_cut,'i', 'lsst', 'r', 'i', 'lsst')

In [ ]:
color_magnitude_diagram(data_set_cut,'vis', 'euclid', 'vis', 'nisp_h', 'euclid')

In [ ]:
color_color_redshift(data_set_cut, 'lsst')

In [ ]:
color_color_redshift(data_set_cut, 'euclid')

In [ ]:
data_set_cut